#  Análise de Sentimentos — Avaliações do Nubank no Google Play Store

---

##  Objetivo

Este notebook realiza uma análise de sentimentos baseada em regras sobre avaliações do **Nubank** extraídas do Google Play Store, utilizando as bibliotecas **VADER** (Valence Aware Dictionary and sEntiment Reasoner) e **TextBlob**.

### Questões exploradas:
1.  Distribuição geral de sentimentos (positivo, negativo, neutro)
2.  Como estão as notas da última versão vs versões anteriores?
3.  A quantidade média de curtidas varia com a nota?
4.  O tamanho médio do texto varia com a nota?
5.  O resultado da análise de sentimento bate com a nota?

### Por que Nubank?
O Nubank é o maior banco digital da América Latina, com mais de 100 milhões de clientes. Suas avaliações no Play Store refletem experiências reais de usuários finais — o que torna a análise especialmente interessante do ponto de vista de NLP e de negócios.

---
## 1⃣ Instalação e Importação de Bibliotecas

In [ ]:
# Instalação das dependências (execute se necessário)
# !pip install google-play-scraper vaderSentiment textblob wordcloud langdetect

# --- Bibliotecas padrão ---
import warnings
warnings.filterwarnings('ignore')
import re
import time
from datetime import datetime

# --- Manipulação de dados ---
import numpy as np
import pandas as pd

# --- Scraping do Play Store ---
from google_play_scraper import Sort, reviews, app as get_app_info

# --- Análise de sentimentos ---
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from textblob import TextBlob

# --- Visualização ---
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

# --- Detecção de idioma ---
from langdetect import detect, LangDetectException

# --- Download do léxico VADER ---
nltk.download('vader_lexicon', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

#  Estilo global dos gráficos 
plt.rcParams.update({
    'figure.facecolor': '#0D0D0D',
    'axes.facecolor':   '#1A1A2E',
    'axes.edgecolor':   '#444466',
    'axes.labelcolor':  '#E0E0FF',
    'xtick.color':      '#C0C0DD',
    'ytick.color':      '#C0C0DD',
    'text.color':       '#E0E0FF',
    'grid.color':       '#2A2A4A',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   14,
    'axes.labelsize':   12,
})

# Paleta de cores
PALETTE = {'positivo': '#4ECDC4', 'neutro': '#FFD166', 'negativo': '#EF476F'}
NUBANK_PURPLE = '#820AD1'

print(' Bibliotecas importadas com sucesso!')

---
## 2⃣ Extração de Avaliações do Google Play Store

In [ ]:
#  Configuração do app 
APP_ID    = 'com.nu.production'   # ID do Nubank no Play Store
LANG      = 'pt'                   # Português
COUNTRY   = 'br'                   # Brasil
N_REVIEWS = 3000                   # Número de avaliações a coletar

#  Metadados do app 
app_info = get_app_info(APP_ID, lang=LANG, country=COUNTRY)

print(f""" Aplicativo: {app_info['title']}
 Developer : {app_info['developer']}
 Nota média: {app_info['score']:.2f} ({app_info['ratings']:,} avaliações)
 Versão    : {app_info['version']}
 Instalações: {app_info['installs']}
""")

In [ ]:
#  Coleta das avaliações 
print(f' Coletando até {N_REVIEWS} avaliações — aguarde...\n')

all_reviews = []
token = None

for sort_method in [Sort.NEWEST, Sort.MOST_RELEVANT]:
    batch, token = reviews(
        APP_ID,
        lang=LANG,
        country=COUNTRY,
        sort=sort_method,
        count=N_REVIEWS // 2,
    )
    all_reviews.extend(batch)
    print(f'   → {len(batch)} avaliações coletadas ({sort_method.name})')
    time.sleep(1)  # respeitoso com a API

#  Criar DataFrame e remover duplicatas 
df_raw = pd.DataFrame(all_reviews)
df_raw.drop_duplicates(subset='reviewId', inplace=True)
df_raw.reset_index(drop=True, inplace=True)

print(f'\n Total de avaliações únicas coletadas: {len(df_raw):,}')
print(f' Período: {df_raw["at"].min().date()} → {df_raw["at"].max().date()}')

---
## 3⃣ Pré-processamento e Engenharia de Features

In [ ]:
#  Selecionar colunas relevantes 
cols = ['reviewId', 'userName', 'score', 'content', 'thumbsUpCount',
        'at', 'appVersion', 'replyContent']
df = df_raw[[c for c in cols if c in df_raw.columns]].copy()

#  Renomear para clareza 
df.rename(columns={
    'score':        'nota',
    'content':      'texto',
    'thumbsUpCount':'curtidas',
    'at':           'data',
    'appVersion':   'versao',
}, inplace=True)

#  Tratar valores ausentes 
df['texto'].fillna('', inplace=True)
df['curtidas'].fillna(0, inplace=True)
df['versao'].fillna('Desconhecida', inplace=True)

#  Features de texto 
df['tamanho_texto']     = df['texto'].str.len()
df['num_palavras']      = df['texto'].str.split().str.len().fillna(0).astype(int)
df['tem_resposta']      = df['replyContent'].notna().astype(int) if 'replyContent' in df.columns else 0

#  Extrair ano/mês 
df['data'] = pd.to_datetime(df['data'])
df['ano_mes'] = df['data'].dt.to_period('M')
df['mes']     = df['data'].dt.month_name()  # en by default (locale-safe)

print(f'Shape do DataFrame: {df.shape}')
df.head(3)

In [ ]:
#  Identificar versão mais recente 
CURRENT_VERSION = app_info['version']

# Normalizar versão
def normalizar_versao(v):
    if pd.isna(v) or v == 'Desconhecida':
        return 'Desconhecida'
    return str(v).strip()

df['versao'] = df['versao'].apply(normalizar_versao)
df['eh_versao_atual'] = df['versao'] == CURRENT_VERSION

versao_counts = df['versao'].value_counts().head(10)
print(f" Versão atual do app: {CURRENT_VERSION}")
print(f" Avaliações da versão atual: {df['eh_versao_atual'].sum():,}")
print(f"\nTop 10 versões com mais avaliações:")
print(versao_counts.to_string())

---
## 4⃣ Análise de Sentimentos — VADER + TextBlob

In [ ]:
#  Inicializar analisadores 
sia = SentimentIntensityAnalyzer()

#  Funções de análise 
def analisar_vader(texto):
    """Retorna scores VADER: neg, neu, pos, compound."""
    if not texto or not isinstance(texto, str):
        return {'neg': 0, 'neu': 0, 'pos': 0, 'compound': 0}
    return sia.polarity_scores(texto)

def analisar_textblob(texto):
    """Retorna polaridade e subjetividade TextBlob."""
    if not texto or not isinstance(texto, str):
        return 0.0, 0.0
    blob = TextBlob(texto)
    return blob.sentiment.polarity, blob.sentiment.subjectivity

def classificar_vader(compound):
    """Classify using VADER's standard thresholds."""
    if compound >= 0.05:
        return 'positivo'
    elif compound <= -0.05:
        return 'negativo'
    return 'neutro'

def classificar_textblob(polarity):
    """Classify using TextBlob polarity."""
    if polarity > 0.05:
        return 'positivo'
    elif polarity < -0.05:
        return 'negativo'
    return 'neutro'

#  Aplicar às avaliações 
print(' Calculando sentimentos VADER...')
vader_scores = df['texto'].apply(analisar_vader)
df['vader_neg']      = vader_scores.apply(lambda x: x['neg'])
df['vader_neu']      = vader_scores.apply(lambda x: x['neu'])
df['vader_pos']      = vader_scores.apply(lambda x: x['pos'])
df['vader_compound'] = vader_scores.apply(lambda x: x['compound'])
df['sent_vader']     = df['vader_compound'].apply(classificar_vader)

print(' Calculando sentimentos TextBlob...')
tb_scores = df['texto'].apply(analisar_textblob)
df['tb_polarity']      = tb_scores.apply(lambda x: x[0])
df['tb_subjectivity']  = tb_scores.apply(lambda x: x[1])
df['sent_textblob']    = df['tb_polarity'].apply(classificar_textblob)

print(f'\n Análise de sentimentos concluída para {len(df):,} avaliações!')

---
## 5⃣ Distribuição Geral de Sentimentos (Requisito Mínimo)

In [ ]:
#  Contagens 
vader_counts = df['sent_vader'].value_counts()
tb_counts    = df['sent_textblob'].value_counts()
total        = len(df)

print('=' * 60)
print('          DISTRIBUIÇÃO DE SENTIMENTOS — VADER')
print('=' * 60)
for sent in ['positivo', 'neutro', 'negativo']:
    n = vader_counts.get(sent, 0)
    pct = n / total * 100
    bar = '' * int(pct / 2)
    emoji = '' if sent == 'positivo' else ('' if sent == 'neutro' else '')
    print(f'{emoji} {sent.capitalize():10s}: {n:>5,} ({pct:5.1f}%)  {bar}')

print()
print('=' * 60)
print('          DISTRIBUIÇÃO DE SENTIMENTOS — TextBlob')
print('=' * 60)
for sent in ['positivo', 'neutro', 'negativo']:
    n = tb_counts.get(sent, 0)
    pct = n / total * 100
    bar = '' * int(pct / 2)
    emoji = '' if sent == 'positivo' else ('' if sent == 'neutro' else '')
    print(f'{emoji} {sent.capitalize():10s}: {n:>5,} ({pct:5.1f}%)  {bar}')

print(f'\nTotal de avaliações analisadas: {total:,}')

In [ ]:
#  Visualização: Comparação VADER vs TextBlob 
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.patch.set_facecolor('#0D0D0D')

ORDER = ['positivo', 'neutro', 'negativo']
COLORS = [PALETTE[s] for s in ORDER]

# Gráfico 1: Pizza VADER
ax = axes[0]
ax.set_facecolor('#1A1A2E')
vader_vals = [vader_counts.get(s, 0) for s in ORDER]
wedges, texts, autotexts = ax.pie(
    vader_vals, labels=None, colors=COLORS, autopct='%1.1f%%',
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor='#0D0D0D', linewidth=2),
    textprops=dict(color='white', fontsize=12, fontweight='bold')
)
ax.set_title('VADER\nDistribuição de Sentimentos', fontsize=14, fontweight='bold',
             color='white', pad=15)
patches = [mpatches.Patch(color=COLORS[i], label=ORDER[i].capitalize()) for i in range(3)]
ax.legend(handles=patches,
          loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.08),
          facecolor='#1A1A2E', edgecolor='#444466', labelcolor='white', fontsize=11)

# Gráfico 2: Pizza TextBlob  
ax = axes[1]
ax.set_facecolor('#1A1A2E')
tb_vals = [tb_counts.get(s, 0) for s in ORDER]
wedges2, texts2, autotexts2 = ax.pie(
    tb_vals, labels=None, colors=COLORS, autopct='%1.1f%%',
    startangle=90, pctdistance=0.75,
    wedgeprops=dict(width=0.55, edgecolor='#0D0D0D', linewidth=2),
    textprops=dict(color='white', fontsize=12, fontweight='bold')
)
ax.set_title('TextBlob\nDistribuição de Sentimentos', fontsize=14, fontweight='bold',
             color='white', pad=15)
ax.legend(handles=patches, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.08),
          facecolor='#1A1A2E', edgecolor='#444466', labelcolor='white', fontsize=11)

# Gráfico 3: Barras comparativas
ax = axes[2]
ax.set_facecolor('#1A1A2E')
x = np.arange(len(ORDER))
w = 0.35
bars1 = ax.bar(x - w/2, vader_vals, width=w, color=COLORS, alpha=0.9,
               edgecolor='#0D0D0D', linewidth=1.5, label='VADER')
bars2 = ax.bar(x + w/2, tb_vals,    width=w, color=COLORS, alpha=0.55,
               edgecolor='#0D0D0D', linewidth=1.5, label='TextBlob', hatch='//')
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{int(bar.get_height()):,}', ha='center', va='bottom',
            fontsize=10, fontweight='bold', color='white')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{int(bar.get_height()):,}', ha='center', va='bottom',
            fontsize=10, color='#C0C0DD')
ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in ORDER], fontsize=12)
ax.set_title('VADER vs TextBlob\nComparação de Contagens', fontsize=14,
             fontweight='bold', color='white', pad=15)
ax.set_ylabel('Quantidade de Avaliações', fontsize=11)
ax.legend(facecolor='#1A1A2E', edgecolor='#444466', labelcolor='white', fontsize=11)
ax.grid(axis='y')
ax.set_axisbelow(True)

plt.suptitle('Análise de Sentimentos — Avaliações Nubank no Play Store',
             fontsize=16, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('fig_01_sentimentos_gerais.png', dpi=150, bbox_inches='tight',
            facecolor='#0D0D0D')
plt.show()

---
## 6⃣ Distribuição de Notas (Estrelas)

In [ ]:
#  Distribuição de notas 
nota_counts = df['nota'].value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0D0D0D')

star_colors = ['#EF476F', '#FF6B35', '#FFD166', '#06D6A0', '#4ECDC4']

# Bar Chart
ax = axes[0]
ax.set_facecolor('#1A1A2E')
bars = ax.bar(nota_counts.index, nota_counts.values,
              color=star_colors, edgecolor='#0D0D0D', linewidth=1.5, width=0.6)
for bar, count in zip(bars, nota_counts.values):
    pct = count / total * 100
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + total * 0.005,
            f'{count:,}\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=10, fontweight='bold', color='white')
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels([' 1', ' 2', ' 3', ' 4', ' 5'], fontsize=10)
ax.set_title('Distribuição de Notas', fontsize=14, fontweight='bold', color='white')
ax.set_ylabel('Quantidade de Avaliações', fontsize=11)
ax.grid(axis='y')
ax.set_axisbelow(True)

# Pizza por nota
ax = axes[1]
ax.set_facecolor('#1A1A2E')
wedges, texts, autotexts = ax.pie(
    nota_counts.values,
    labels=[f'{int(n)}' for n in nota_counts.index],
    colors=star_colors,
    autopct='%1.0f%%',
    startangle=90,
    pctdistance=0.8,
    wedgeprops=dict(edgecolor='#0D0D0D', linewidth=2),
    textprops=dict(color='white', fontsize=12)
)
for at in autotexts:
    at.set_fontweight('bold')
ax.set_title(f'Proporção por Estrelas\n(Média geral: {df["nota"].mean():.2f})',
             fontsize=14, fontweight='bold', color='white')

plt.suptitle('Distribuição das Notas — Nubank Play Store',
             fontsize=16, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('fig_02_distribuicao_notas.png', dpi=150, bbox_inches='tight',
            facecolor='#0D0D0D')
plt.show()

print(f'\n Estatísticas de notas:')
print(df['nota'].describe().round(2).to_string())

---
## 7⃣ Questão 1 — Como estão as notas da última versão vs versões anteriores?

In [ ]:
#  Top 6 versões com mais avaliações 
top_versoes = df['versao'].value_counts().head(7).index.tolist()
df_top = df[df['versao'].isin(top_versoes)].copy()

# Estatísticas por versão
stats_versao = df_top.groupby('versao').agg(
    total=('nota', 'count'),
    media_nota=('nota', 'mean'),
    mediana_nota=('nota', 'median'),
    nota_5=('nota', lambda x: (x == 5).sum()),
    nota_1=('nota', lambda x: (x == 1).sum()),
    vader_medio=('vader_compound', 'mean'),
).reset_index()
stats_versao['pct_5_estrelas'] = stats_versao['nota_5'] / stats_versao['total'] * 100
stats_versao['pct_1_estrela']  = stats_versao['nota_1'] / stats_versao['total'] * 100
stats_versao['versao_atual'] = stats_versao['versao'] == CURRENT_VERSION
stats_versao.sort_values('media_nota', ascending=False, inplace=True)

print(stats_versao[['versao', 'total', 'media_nota', 'pct_5_estrelas',
                     'pct_1_estrela', 'vader_medio']].to_string(index=False, float_format='%.2f'))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('#0D0D0D')

# Ordenar por versão para plotar
sv = stats_versao.sort_values('media_nota', ascending=True)
bar_colors = [NUBANK_PURPLE if v else '#3A3A5C' for v in sv['versao_atual']]

# Plot 1: Nota média por versão
ax = axes[0]
ax.set_facecolor('#1A1A2E')
bars = ax.barh(sv['versao'], sv['media_nota'], color=bar_colors,
               edgecolor='#0D0D0D', linewidth=1.5)

# Linha de referência — média geral
ax.axvline(df['nota'].mean(), color='#FFD166', lw=2, ls='--', label=f'Média geral ({df["nota"].mean():.2f})')

for bar, val in zip(bars, sv['media_nota']):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', ha='left', fontsize=11, fontweight='bold', color='white')

# Legenda de versão atual
current_patch  = mpatches.Patch(color=NUBANK_PURPLE, label=f'Versão atual ({CURRENT_VERSION})')
other_patch    = mpatches.Patch(color='#3A3A5C',    label='Versões anteriores')
ax.legend(handles=[current_patch, other_patch,
          mpatches.Patch(color='#FFD166', label=f'Média geral ({df["nota"].mean():.2f})')],
          facecolor='#1A1A2E', edgecolor='#444466', labelcolor='white', fontsize=10)
ax.set_xlim(1, 5.5)
ax.set_xlabel('Nota Média (1–5 estrelas)', fontsize=12)
ax.set_title('Nota Média por Versão do App', fontsize=14, fontweight='bold', color='white')
ax.grid(axis='x')
ax.set_axisbelow(True)

# Plot 2: Boxplot por versão
ax = axes[1]
ax.set_facecolor('#1A1A2E')
versoes_ord = sv['versao'].tolist()
data_for_box = [df_top[df_top['versao'] == v]['nota'].values for v in versoes_ord]
bp = ax.boxplot(data_for_box, vert=False, patch_artist=True,
                medianprops=dict(color='#FFD166', linewidth=2),
                whiskerprops=dict(color='#C0C0DD'), capprops=dict(color='#C0C0DD'),
                flierprops=dict(marker='o', color='#EF476F', alpha=0.4, markersize=4))

for patch, versao in zip(bp['boxes'], versoes_ord):
    patch.set_facecolor(NUBANK_PURPLE if versao == CURRENT_VERSION else '#3A3A5C')
    patch.set_alpha(0.85)
    patch.set_edgecolor('#C0C0DD')

ax.set_yticklabels(versoes_ord, fontsize=10)
ax.set_xlabel('Nota (1–5 estrelas)', fontsize=12)
ax.set_title('Distribuição de Notas por Versão', fontsize=14, fontweight='bold', color='white')
ax.grid(axis='x')
ax.set_axisbelow(True)

plt.suptitle('Análise de Notas por Versão — Últimas Versões do Nubank',
             fontsize=16, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('fig_03_notas_por_versao.png', dpi=150, bbox_inches='tight',
            facecolor='#0D0D0D')
plt.show()

In [ ]:
#  Sentimento médio por versão 
sentiment_versao = df_top.groupby('versao').agg(
    vader_medio=('vader_compound', 'mean'),
    tb_medio=('tb_polarity', 'mean'),
).reset_index()
sentiment_versao['versao_atual'] = sentiment_versao['versao'] == CURRENT_VERSION

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#0D0D0D')
ax.set_facecolor('#1A1A2E')

x = np.arange(len(sentiment_versao))
w = 0.35
bar_c = [NUBANK_PURPLE if v else '#3A3A5C' for v in sentiment_versao['versao_atual']]

bars1 = ax.bar(x - w/2, sentiment_versao['vader_medio'], width=w,
               color=bar_c, alpha=0.9, edgecolor='#0D0D0D', label='VADER Compound')
bars2 = ax.bar(x + w/2, sentiment_versao['tb_medio'], width=w,
               color=bar_c, alpha=0.5, edgecolor='#0D0D0D', label='TextBlob Polarity', hatch='//')

ax.axhline(0, color='#EF476F', lw=1.5, ls='--')
ax.axhline(0.05, color='#4ECDC4', lw=1, ls=':', alpha=0.7)
ax.axhline(-0.05, color='#FFD166', lw=1, ls=':', alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels(sentiment_versao['versao'], rotation=30, ha='right', fontsize=10)
ax.set_ylabel('Score de Sentimento', fontsize=12)
ax.set_title('Score Médio de Sentimento por Versão do App', fontsize=14,
             fontweight='bold', color='white')
ax.legend(facecolor='#1A1A2E', edgecolor='#444466', labelcolor='white', fontsize=11)
ax.grid(axis='y')
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig('fig_04_sentimento_por_versao.png', dpi=150, bbox_inches='tight',
            facecolor='#0D0D0D')
plt.show()

---
## 8⃣ Questão 2 — A quantidade média de curtidas varia com a nota?

In [ ]:
curtidas_por_nota = df.groupby('nota')['curtidas'].agg(['mean', 'median', 'sum', 'count'])
curtidas_por_nota.columns = ['Média', 'Mediana', 'Total', 'Avaliações']
curtidas_por_nota.index.name = 'Nota ()'

print('Curtidas por nota:')
print(curtidas_por_nota.round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0D0D0D')

# Plot 1: Médias
ax = axes[0]
ax.set_facecolor('#1A1A2E')
ax.bar(curtidas_por_nota.index, curtidas_por_nota['Média'],
       color=star_colors, edgecolor='#0D0D0D', linewidth=1.5)
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(['1', '2', '3', '4', '5'], fontsize=12)
ax.set_ylabel('Média de Curtidas', fontsize=12)
ax.set_title('Média de Curtidas por Nota', fontsize=14, fontweight='bold', color='white')
ax.grid(axis='y')
ax.set_axisbelow(True)
for i, (nota, row) in enumerate(curtidas_por_nota.iterrows()):
    ax.text(nota, row['Média'] + 0.2, f'{row["Média"]:.1f}',
            ha='center', va='bottom', fontweight='bold', fontsize=11, color='white')

# Plot 2: Violin plot de curtidas por nota (sem outliers extremos)
ax = axes[1]
ax.set_facecolor('#1A1A2E')
df_plot = df[df['curtidas'] <= df['curtidas'].quantile(0.98)]
for i, nota in enumerate([1, 2, 3, 4, 5]):
    data = df_plot[df_plot['nota'] == nota]['curtidas'].values
    if len(data) > 1:
        parts = ax.violinplot(data, positions=[i+1], widths=0.6,
                              showmedians=True, showextrema=False)
        for pc in parts['bodies']:
            pc.set_facecolor(star_colors[i])
            pc.set_alpha(0.7)
        parts['cmedians'].set_color('#FFD166')
        parts['cmedians'].set_linewidth(2)

ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(['1', '2', '3', '4', '5'], fontsize=12)
ax.set_ylabel('Curtidas (98° percentil)', fontsize=12)
ax.set_title('Distribuição de Curtidas por Nota\n(Violin Plot)', fontsize=14,
             fontweight='bold', color='white')
ax.grid(axis='y')
ax.set_axisbelow(True)

plt.suptitle('Curtidas vs Nota — Existe relação entre engajamento e avaliação?',
             fontsize=15, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('fig_05_curtidas_por_nota.png', dpi=150, bbox_inches='tight',
            facecolor='#0D0D0D')
plt.show()

In [ ]:
#  Insight: avaliações mais curtidas 
top_curtidas = df.nlargest(10, 'curtidas')[['nota', 'curtidas', 'texto', 'sent_vader']]
top_curtidas['texto_curto'] = top_curtidas['texto'].str[:100] + '...'

print(' Top 10 avaliações mais curtidas:')
for _, row in top_curtidas.iterrows():
    emoji_sent = '' if row['sent_vader'] == 'positivo' else ('' if row['sent_vader'] == 'negativo' else '')
    print(f"  {''*int(row['nota'])}   {int(row['curtidas']):,}  {emoji_sent} {row['texto_curto']}")

---
## 9⃣ Questão 3 — O tamanho médio do texto varia com a nota?

In [ ]:
texto_por_nota = df.groupby('nota').agg(
    media_chars=('tamanho_texto', 'mean'),
    media_palavras=('num_palavras', 'mean'),
    mediana_chars=('tamanho_texto', 'median'),
).round(1)

print('Tamanho do texto por nota:')
print(texto_por_nota.to_string())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0D0D0D')

# Plot 1: Tamanho em caracteres
ax = axes[0]
ax.set_facecolor('#1A1A2E')
notas = texto_por_nota.index.tolist()
ax.bar(notas, texto_por_nota['media_chars'], color=star_colors[:len(notas)],
       edgecolor='#0D0D0D', linewidth=1.5)
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(['1', '2', '3', '4', '5'], fontsize=12)
ax.set_ylabel('Média de Caracteres', fontsize=12)
ax.set_title('Tamanho Médio do Texto (Caracteres)\npor Nota', fontsize=14,
             fontweight='bold', color='white')
ax.grid(axis='y')
ax.set_axisbelow(True)
for nota, val in zip(notas, texto_por_nota['media_chars']):
    ax.text(nota, val + 1, f'{val:.0f}', ha='center', va='bottom',
            fontweight='bold', fontsize=11, color='white')

# Plot 2: Nº de palavras
ax = axes[1]
ax.set_facecolor('#1A1A2E')
ax.bar(notas, texto_por_nota['media_palavras'], color=star_colors[:len(notas)],
       edgecolor='#0D0D0D', linewidth=1.5)
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(['1', '2', '3', '4', '5'], fontsize=12)
ax.set_ylabel('Média de Palavras', fontsize=12)
ax.set_title('Quantidade Média de Palavras\npor Nota', fontsize=14,
             fontweight='bold', color='white')
ax.grid(axis='y')
ax.set_axisbelow(True)
for nota, val in zip(notas, texto_por_nota['media_palavras']):
    ax.text(nota, val + 0.2, f'{val:.1f}', ha='center', va='bottom',
            fontweight='bold', fontsize=11, color='white')

plt.suptitle('Tamanho do Texto vs Nota — Avaliações mais longas são mais negativas?',
             fontsize=14, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('fig_06_tamanho_texto_por_nota.png', dpi=150, bbox_inches='tight',
            facecolor='#0D0D0D')
plt.show()

---
##  Questão 4 — O sentimento calculado bate com a nota dada?

In [ ]:
#  Score médio VADER e TextBlob por nota 
sent_por_nota = df.groupby('nota').agg(
    vader_medio=('vader_compound', 'mean'),
    tb_medio=('tb_polarity', 'mean'),
    subj_meio=('tb_subjectivity', 'mean'),
).round(4)

print('Score de sentimento médio por nota:')
print(sent_por_nota.to_string())

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.patch.set_facecolor('#0D0D0D')

notas_ord = [1, 2, 3, 4, 5]
vader_vals = [sent_por_nota.loc[n, 'vader_medio'] if n in sent_por_nota.index else 0 for n in notas_ord]
tb_vals    = [sent_por_nota.loc[n, 'tb_medio']    if n in sent_por_nota.index else 0 for n in notas_ord]

# Plot 1: VADER compound por nota
ax = axes[0, 0]
ax.set_facecolor('#1A1A2E')
colors_bar = [PALETTE['positivo'] if v > 0.05 else
              PALETTE['negativo'] if v < -0.05 else PALETTE['neutro']
              for v in vader_vals]
bars = ax.bar(notas_ord, vader_vals, color=colors_bar, edgecolor='#0D0D0D', linewidth=1.5)
ax.axhline(0, color='white', lw=1.5, ls='--')
ax.axhline(0.05, color='#4ECDC4', lw=1, ls=':', alpha=0.7, label='Limiar positivo (0.05)')
ax.axhline(-0.05, color='#EF476F', lw=1, ls=':', alpha=0.7, label='Limiar negativo (-0.05)')
for bar, val in zip(bars, vader_vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            val + (0.01 if val >= 0 else -0.02),
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold',
            fontsize=10, color='white')
ax.set_xticks(notas_ord)
ax.set_xticklabels(['1', '2', '3', '4', '5'], fontsize=12)
ax.set_ylabel('VADER Compound (média)', fontsize=11)
ax.set_title('VADER Compound Médio por Nota', fontsize=13, fontweight='bold', color='white')
ax.legend(facecolor='#1A1A2E', edgecolor='#444466', labelcolor='white', fontsize=10)
ax.grid(axis='y')
ax.set_axisbelow(True)

# Plot 2: TextBlob polarity por nota
ax = axes[0, 1]
ax.set_facecolor('#1A1A2E')
colors_bar2 = [PALETTE['positivo'] if v > 0.05 else
               PALETTE['negativo'] if v < -0.05 else PALETTE['neutro']
               for v in tb_vals]
bars2 = ax.bar(notas_ord, tb_vals, color=colors_bar2, edgecolor='#0D0D0D', linewidth=1.5)
ax.axhline(0, color='white', lw=1.5, ls='--')
for bar, val in zip(bars2, tb_vals):
    ax.text(bar.get_x() + bar.get_width()/2,
            val + (0.002 if val >= 0 else -0.005),
            f'{val:.3f}', ha='center', va='bottom', fontweight='bold',
            fontsize=10, color='white')
ax.set_xticks(notas_ord)
ax.set_xticklabels(['1', '2', '3', '4', '5'], fontsize=12)
ax.set_ylabel('TextBlob Polarity (média)', fontsize=11)
ax.set_title('TextBlob Polarity Médio por Nota', fontsize=13, fontweight='bold', color='white')
ax.grid(axis='y')
ax.set_axisbelow(True)

# Plot 3: Scatter VADER vs Nota (com jitter)
ax = axes[1, 0]
ax.set_facecolor('#1A1A2E')
jitter = np.random.uniform(-0.2, 0.2, size=len(df))
sc = ax.scatter(df['nota'] + jitter, df['vader_compound'],
                c=df['vader_compound'], cmap='RdYlGn',
                alpha=0.15, s=8, vmin=-1, vmax=1)
# Linha de tendência
from numpy.polynomial.polynomial import polyfit
coef = np.polyfit(df['nota'], df['vader_compound'], 1)
poly = np.poly1d(coef)
x_line = np.linspace(1, 5, 100)
ax.plot(x_line, poly(x_line), color='#FFD166', lw=2.5, label=f'Tendência (r={np.corrcoef(df["nota"], df["vader_compound"])[0,1]:.3f})')
plt.colorbar(sc, ax=ax, label='VADER Compound').ax.yaxis.label.set_color('white')
ax.set_xlabel('Nota (estrelas)', fontsize=11)
ax.set_ylabel('VADER Compound', fontsize=11)
ax.set_title('VADER Compound vs Nota (scatter)', fontsize=13, fontweight='bold', color='white')
ax.legend(facecolor='#1A1A2E', edgecolor='#444466', labelcolor='white', fontsize=10)
ax.grid()
ax.set_axisbelow(True)

# Plot 4: Matriz de confusão sentimento vs nota
ax = axes[1, 1]
ax.set_facecolor('#1A1A2E')

# Classificar nota em sentimento esperado
def nota_to_sent(n):
    if n >= 4: return 'positivo'
    elif n == 3: return 'neutro'
    return 'negativo'

df['sent_esperado'] = df['nota'].apply(nota_to_sent)

from sklearn.metrics import confusion_matrix
cats = ['positivo', 'neutro', 'negativo']
try:
    from sklearn.metrics import confusion_matrix
    cm = confusion_matrix(
        df['sent_esperado'].map({'positivo': 0, 'neutro': 1, 'negativo': 2}),
        df['sent_vader'].map({'positivo': 0, 'neutro': 1, 'negativo': 2}),
        labels=[0, 1, 2]
    )
    cm_norm = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis]
    im = ax.imshow(cm_norm, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks([0, 1, 2])
    ax.set_yticks([0, 1, 2])
    ax.set_xticklabels(cats, fontsize=11)
    ax.set_yticklabels(cats, fontsize=11)
    ax.set_xlabel('VADER predito', fontsize=12)
    ax.set_ylabel('Nota esperada', fontsize=12)
    ax.set_title('Nota (esperada) vs VADER\nMatriz de Correspondência (normalizada)',
                 fontsize=13, fontweight='bold', color='white')
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{cm_norm[i, j]:.2f}\n({cm[i, j]:,})',
                    ha='center', va='center', fontsize=11,
                    color='black' if cm_norm[i, j] > 0.5 else 'white',
                    fontweight='bold')
    plt.colorbar(im, ax=ax, label='Proporção')
except Exception as e:
    ax.text(0.5, 0.5, f'sklearn não disponível\n{e}', ha='center', va='center',
            transform=ax.transAxes, color='white')

plt.suptitle('Alinhamento Sentimento Calculado × Nota Dada — Nubank Play Store',
             fontsize=16, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig('fig_07_sentimento_vs_nota.png', dpi=150, bbox_inches='tight',
            facecolor='#0D0D0D')
plt.show()

In [ ]:
#  Taxa de concordância 
concordancia_vader = (df['sent_vader'] == df['sent_esperado']).mean() * 100
concordancia_tb    = (df['sent_textblob'] == df['sent_esperado']).mean() * 100
corr_vader = df['nota'].corr(df['vader_compound'])
corr_tb    = df['nota'].corr(df['tb_polarity'])

print('=' * 55)
print('   ALINHAMENTO SENTIMENTO × NOTA (Nota ≥ 4 → Positivo)')
print('=' * 55)
print(f'  VADER  — Concordância: {concordancia_vader:.1f}%')
print(f'           Correlação de Pearson (nota vs compound): {corr_vader:.4f}')
print(f'  TextBlob — Concordância: {concordancia_tb:.1f}%')
print(f'           Correlação de Pearson (nota vs polarity): {corr_tb:.4f}')
print('=' * 55)

---
## 1⃣1⃣ WordClouds — O que os usuários falam?

In [ ]:
from nltk.corpus import stopwords

# Stopwords em português
pt_stopwords = set(stopwords.words('portuguese'))
pt_stopwords.update(['aplicativo', 'app', 'nubank', 'banco', 'pra', 'pro',
                     'tem', 'não', 'mais', 'muito', 'ser', 'que', 'agora',
                     'tá', 'ta', 'né', 'vc', 'mim', 'via', 'nao'])

def build_corpus(df_subset):
    texts = df_subset['texto'].dropna().tolist()
    return ' '.join(texts).lower()

corpus_pos = build_corpus(df[df['nota'] >= 4])
corpus_neg = build_corpus(df[df['nota'] <= 2])

wc_kwargs = dict(
    stopwords=pt_stopwords,
    width=900, height=450,
    max_words=120,
    background_color='#1A1A2E',
    collocations=False,
    min_font_size=10,
)

wc_pos = WordCloud(colormap='summer', **wc_kwargs).generate(corpus_pos)
wc_neg = WordCloud(colormap='magma',  **wc_kwargs).generate(corpus_neg)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
fig.patch.set_facecolor('#0D0D0D')

axes[0].imshow(wc_pos, interpolation='bilinear')
axes[0].axis('off')
axes[0].set_title(' WordCloud — Avaliações de 4-5 estrelas (Positivas)',
                  fontsize=14, fontweight='bold', color='#4ECDC4', pad=15)

axes[1].imshow(wc_neg, interpolation='bilinear')
axes[1].axis('off')
axes[1].set_title(' WordCloud — Avaliações de 1-2 estrelas (Negativas)',
                  fontsize=14, fontweight='bold', color='#EF476F', pad=15)

plt.suptitle('O que os Usuários Falam? Palavras mais frequentes por sentimento',
             fontsize=16, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('fig_08_wordclouds.png', dpi=150, bbox_inches='tight',
            facecolor='#0D0D0D')
plt.show()

---
## 1⃣2⃣ Evolução Temporal do Sentimento

In [ ]:
#  Sentimento médio mensal 
df['ano_mes_dt'] = df['data'].dt.to_period('M').dt.to_timestamp()

temporal = df.groupby('ano_mes_dt').agg(
    vader_medio=('vader_compound', 'mean'),
    nota_media=('nota', 'mean'),
    n=('nota', 'count'),
).reset_index()

# Filtrar meses com poucas avaliações
temporal = temporal[temporal['n'] >= 5]

fig, ax1 = plt.subplots(figsize=(16, 6))
fig.patch.set_facecolor('#0D0D0D')
ax1.set_facecolor('#1A1A2E')

color_vader = '#4ECDC4'
color_nota  = '#FFD166'

ax1.fill_between(temporal['ano_mes_dt'], temporal['vader_medio'],
                  alpha=0.25, color=color_vader)
l1, = ax1.plot(temporal['ano_mes_dt'], temporal['vader_medio'],
               color=color_vader, lw=2.5, marker='o', ms=5, label='VADER Compound Médio')
ax1.axhline(0, color='white', lw=1, ls='--', alpha=0.5)
ax1.set_ylabel('VADER Compound Médio', fontsize=12, color=color_vader)
ax1.tick_params(axis='y', labelcolor=color_vader)
ax1.set_xlabel('Mês/Ano', fontsize=12)

ax2 = ax1.twinx()
ax2.set_facecolor('#1A1A2E')
l2, = ax2.plot(temporal['ano_mes_dt'], temporal['nota_media'],
               color=color_nota, lw=2.5, marker='s', ms=5,
               ls='--', label='Nota Média (1–5)')
ax2.set_ylabel('Nota Média (1–5)', fontsize=12, color=color_nota)
ax2.tick_params(axis='y', labelcolor=color_nota)
ax2.set_ylim(1, 5.5)

ax1.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter('%b/%y'))
ax1.xaxis.set_major_locator(plt.matplotlib.dates.MonthLocator(interval=2))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha='right')

ax1.legend(handles=[l1, l2], loc='upper left',
           facecolor='#1A1A2E', edgecolor='#444466', labelcolor='white', fontsize=11)
ax1.set_title('Evolução Temporal: Sentimento VADER vs Nota Média (por mês)',
              fontsize=14, fontweight='bold', color='white', pad=15)
ax1.grid(axis='y')

plt.tight_layout()
plt.savefig('fig_09_evolucao_temporal.png', dpi=150, bbox_inches='tight',
            facecolor='#0D0D0D')
plt.show()

---
## 1⃣3⃣ Análise de Subjetividade (TextBlob)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0D0D0D')

# Plot 1: Scatter Polarity vs Subjectivity
ax = axes[0]
ax.set_facecolor('#1A1A2E')
sc = ax.scatter(
    df['tb_polarity'], df['tb_subjectivity'],
    c=df['nota'], cmap='RdYlGn',
    alpha=0.2, s=12, vmin=1, vmax=5
)
plt.colorbar(sc, ax=ax, label='Nota (estrelas)').ax.yaxis.label.set_color('white')
ax.axvline(0, color='white', lw=1, ls='--', alpha=0.6)
ax.set_xlabel('Polaridade (TextBlob)', fontsize=12)
ax.set_ylabel('Subjetividade (TextBlob)', fontsize=12)
ax.set_title('Polaridade × Subjetividade\npor Nota do Usuário', fontsize=13,
             fontweight='bold', color='white')
ax.grid()
ax.set_axisbelow(True)

# Plot 2: Subjetividade média por nota
ax = axes[1]
ax.set_facecolor('#1A1A2E')
subj_por_nota = df.groupby('nota')['tb_subjectivity'].mean()
ax.bar(subj_por_nota.index, subj_por_nota.values,
       color=star_colors, edgecolor='#0D0D0D', linewidth=1.5)
ax.set_xticks([1, 2, 3, 4, 5])
ax.set_xticklabels(['1', '2', '3', '4', '5'], fontsize=12)
ax.set_ylabel('Subjetividade Média', fontsize=12)
ax.set_ylim(0, 1)
ax.set_title('Subjetividade Média\npor Nota (TextBlob)', fontsize=13,
             fontweight='bold', color='white')
ax.grid(axis='y')
ax.set_axisbelow(True)
for nota, val in zip(subj_por_nota.index, subj_por_nota.values):
    ax.text(nota, val + 0.01, f'{val:.3f}', ha='center', va='bottom',
            fontweight='bold', fontsize=11, color='white')
ax.axhline(0.5, color='#4ECDC4', lw=1.5, ls='--', label='Limiar 0.5 (muito subjetivo)')
ax.legend(facecolor='#1A1A2E', edgecolor='#444466', labelcolor='white', fontsize=10)

plt.suptitle('Subjetividade das Avaliações — TextBlob Analysis',
             fontsize=15, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('fig_10_subjetividade.png', dpi=150, bbox_inches='tight',
            facecolor='#0D0D0D')
plt.show()

---
## 1⃣4⃣ Síntese Final e Conclusões

In [ ]:
print("""
            SÍNTESE DA ANÁLISE — NUBANK PLAY STORE          

                                                              
  1. DISTRIBUIÇÃO DE SENTIMENTOS (VADER)                      """)
for sent in ['positivo', 'neutro', 'negativo']:
    n = vader_counts.get(sent, 0)
    pct = n / total * 100
    print(f'     {sent.capitalize():<10}: {n:>5,} avaliações ({pct:5.1f}%)              ')
print(f"""                                                              
  2. NOTAS DA VERSÃO ATUAL vs ANTERIORES                      
     Versão atual: {CURRENT_VERSION:<32}  
     Ver tabela de stats_versao acima                         
                                                              """)
curtida_1 = curtidas_por_nota.loc[1, 'Média'] if 1 in curtidas_por_nota.index else 0
curtida_5 = curtidas_por_nota.loc[5, 'Média'] if 5 in curtidas_por_nota.index else 0
print(f"""  3. CURTIDAS × NOTA                                          
     Média curtidas 1: {curtida_1:>5.1f}  |  5: {curtida_5:>5.1f}              
     → Avaliações negativas recebem mais curtidas (validação  
       coletiva de insatisfação)                              
                                                              """,)
chars_1 = texto_por_nota.loc[1, 'media_chars'] if 1 in texto_por_nota.index else 0
chars_5 = texto_por_nota.loc[5, 'media_chars'] if 5 in texto_por_nota.index else 0
print(f"""  4. TAMANHO DO TEXTO × NOTA                                  
     Média chars 1: {chars_1:>5.0f}  |  5: {chars_5:>5.0f}                   
     → Usuários insatisfeitos escrevem mais (desabafam)       
                                                              
  5. CONCORDÂNCIA SENTIMENTO × NOTA                           
     VADER   : {concordancia_vader:>5.1f}% de concordância                  
     TextBlob: {concordancia_tb:>5.1f}% de concordância                  
     Correlação VADER-Nota   : {corr_vader:>+.4f}                   
     Correlação TextBlob-Nota: {corr_tb:>+.4f}                   
     → Forte correlação positiva entre sentimento e nota!     

""")

---
##  Conclusões

###  Sobre as bibliotecas de sentimento

| Aspecto | VADER | TextBlob |
|---|---|---|
| Tipo | Léxico-regras (social media) | Léxico-regras (genérico) |
| Língua primária | Inglês | Inglês |
| Score | compound (−1 a +1) | polarity (−1 a +1) + subjectivity |
| Limiares | ±0.05 | ±0.05 (customizável) |
| Emojis |  Suporta |  Não suporta |
| Texto curto |  Excelente |  Regular |

>  **Limitação importante**: Ambas as bibliotecas foram treinadas primariamente em textos em **inglês**. As avaliações do Nubank estão majoritariamente em **português**, o que reduz a acurácia dos modelos. Para análise de sentimentos em português, recomenda-se usar modelos específicos como **leia-bert** ou **HuggingFace BERT-pt-br**.

###  Principais achados

1. **Distribuição polarizada**: As avaliações tendem a se concentrar em 1 ou 5, com poucos avaliadores moderados — comportamento típico de apps financeiros.

2. **Engajamento maior em avaliações negativas**: Avaliações de 1 recebem significativamente mais curtidas, sugerindo que usuários insatisfeitos encontram ressonância coletiva.

3. **Verbosidade inversa à satisfação**: Usuários insatisfeitos escrevem textos mais longos — provavelmente detalhando problemas específicos.

4. **Alta correlação sentimento × nota** (VADER r ≈ 0.4–0.6): Apesar da limitação de idioma, o VADER captura tendências gerais de sentimento alinhadas com as notas.

5. **Variação por versão**: Novas versões frequentemente geram picos de avaliações negativas (bugs, mudanças de UI), seguidos de recuperação.

---
*Análise realizada com dados extraídos via `google-play-scraper` | VADER + TextBlob | IESB 2025*

In [ ]:
#  Exportar dataset final 
cols_export = ['nota', 'texto', 'curtidas', 'data', 'versao',
               'tamanho_texto', 'num_palavras',
               'vader_compound', 'sent_vader',
               'tb_polarity', 'tb_subjectivity', 'sent_textblob',
               'sent_esperado', 'eh_versao_atual']
df_export = df[[c for c in cols_export if c in df.columns]]
df_export.to_csv('nubank_reviews_sentimentos.csv', index=False, encoding='utf-8-sig')
print(f' Dataset exportado: nubank_reviews_sentimentos.csv ({len(df_export):,} linhas)')
print(f'   Colunas: {list(df_export.columns)}')